# Documentation
**Author:** Spencer Ressel

**Created:** August 16th, 2024

***

This notebook analyzes aqua-planet simulation data from the CAM6 model run by Mu-Ting Chien. 
Specifically, MJO diagnostics are computed following the specifications listed by the CLIVAR Madden-Julian Oscillation Working Group (MJOWG).
For details, see https://atmos.uw.edu/~daehyun/mjo_diagnostics/

***

# Imports

In [4]:
%load_ext autoreload
%autoreload 2

# Aquaplanet analysis config file
import config
import coords

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)
logger.info("Loading Imports...")

# File management
import glob
import os
import sys
from datetime import datetime, timedelta
import copy
import cftime

# Data anaylsis
import numpy as np
import scipy
import xarray as xr
xr.set_options(keep_attrs=True)
import scipy.signal as signal
from scipy.stats import t

# Plotting
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import ticker as mticker
from matplotlib.gridspec import GridSpec
from cartopy import util as cutil
import string

# Auxiliary functions
from load_aquaplanet_data import *
from processing_functions import *

from auxiliary_functions.one_two_one_filter import one_two_one_filter
from auxiliary_functions.plotting_utils import bmh_colors, modified_colormap, set_plot_mode, get_figsize

logger.info("Imports loaded")

2026-08-04 13:28:47,307 [INFO] Loading Imports...
/glade/u/home/sressel/thesis-work/python/auxiliary_functions/src/auxiliary_functions/__init__.py:49: UserWarning: Could not import from calculate_velocity_potential (likely missing 'windspharm'): No module named 'windspharm'
  warnings.warn(f"Could not import from calculate_velocity_potential (likely missing 'windspharm'): {e}")
2026-08-04 13:28:47,510 [INFO] Imports loaded


# Load processed data

## Detrended data

In [6]:
print(f"Loading detrended data")
reload_subset = False

if not 'multi_experiment_variables_detrended' in locals():
    multi_experiment_variables_detrended = load_multi_experiment_processed_data(
        ['Precipitation'],
        'detrended'
    )
else:
    multi_experiment_variables_detrended = load_multi_experiment_processed_data(
        ['Precipitation'],
        'detrended',
        multi_experiment_variables_detrended,
        False
    )

2026-08-04 13:29:04,057 [INFO] Loading detrended data
2026-08-04 13:29:04,058 [INFO] (1/1) Precipitation...
2026-08-04 13:29:04,059 [INFO]     Experiment: -4K...
2026-08-04 13:29:04,136 [INFO]     Experiment: 0K...
2026-08-04 13:29:04,193 [INFO]     Experiment: 4K...


Loading detrended data


2026-08-04 13:29:05,245 [INFO] Finished


# Space-Time Decompositions

## Power Spectra

### Calculate space-time power spectra

In [7]:
str_width = 40
xr.set_options(keep_attrs=True)
print(f"{'Space-Time Power Spectra':^{str_width}}")

max_latitude = 33
mask_land = False

recalculate_signal = True
plot_raw_spectrum = False
plot_background_spectrum = True
plot_signal_strength = True
save_plots = False
n_smooths = 15

# if 'symmetric_power_spectrum' not in locals():
symmetric_signal_fft = {}
asymmetric_signal_fft = {}
raw_symmetric_power = {}
raw_asymmetric_power = {}
background_spectrum = {}
symmetric_power_spectrum = {}
asymmetric_power_spectrum = {}

# Subset into segments in time (96 days, overlap 60 days)
segment_length = 192
# overlap = 192//2
# segment_length = 96
overlap = 96
window_width = 5

for index, variable in enumerate([
    multi_experiment_variables_detrended['Precipitation'],
]):

    variable_id = f"{variable.attrs['file_id']}{(str(variable.plev.values) if 'plev' in variable.coords else '')}"
    if variable_id in symmetric_signal_fft and recalculate_signal == False:
        continue

    print(f"{'='*str_width}")
    print(f"{f'Variable: {variable_id}':^{str_width}}")
    print(f"{'-'*str_width}")

    # Calculate the frequency and zonal wavenumber axis coordinates
    frequency = np.arange(-segment_length / 2, segment_length / 2) * 1 / segment_length
    zonal_wavenumber = (
        np.arange(-len(variable.lon) / 2, len(variable.lon) / 2)
        * (1 / 2.5)
        / len(variable.lon)
        * 360
    )

    symmetric_power_spectrum[variable_id] = []
    asymmetric_power_spectrum[variable_id] = []

    for experiment_index, experiment in enumerate(coords.experiments.values):
        print(f"{f'Experiment: {experiment}':<{str_width}}")
        print(f"{f'→ Defining symmetric signals...':<{str_width-1}}", end="")

        experiment_variable = variable.sel(experiment=experiment)
        # Separate into symmetric/antisymmetric component
        symmetric_signal = xr.zeros_like(experiment_variable.sel(lat=slice(0, max_latitude)))
        asymmetric_signal = xr.zeros_like(experiment_variable.sel(lat=slice(0, max_latitude)))

        for latitude_index, latitude in enumerate(symmetric_signal.lat):
            symmetric_signal[:, latitude_index, :] = (1 / 2) * (
                experiment_variable.sel(lat=latitude, method="nearest")
                + experiment_variable.sel(lat=-latitude, method="nearest")
            )
            asymmetric_signal[:, latitude_index, :] = -(1 / 2) * (
                experiment_variable.sel(lat=latitude, method="nearest")
                - experiment_variable.sel(lat=-latitude, method="nearest")
            )

        symmetric_signal = symmetric_signal.fillna(0)
        asymmetric_signal = asymmetric_signal.fillna(0)

        # Define the Hann window
        hann_window = np.concatenate(
            (
                np.hanning(2*window_width)[:window_width],
                np.ones(segment_length - window_width * 2),
                np.hanning(2*window_width)[::-1][window_width:],
            ),
            axis=0,
        )
        print(rf"{'✔':>1}")

        print(f"{f'→ Segmenting data...':<{str_width-1}}", end="")
        # Segment the data
        symmetric_signal_segmented = symmetric_signal.rolling(
            time=segment_length, center=False
        ).construct("day_in_segment").rename({'time': 'segment_index'})

        symmetric_signal_segmented = symmetric_signal_segmented[(segment_length - 1) :][
            :: (segment_length - overlap)
        ].transpose("segment_index", "day_in_segment", "lat", "lon")

        asymmetric_signal_segmented = asymmetric_signal.rolling(
            time=segment_length, center=False
        ).construct("day_in_segment").rename({'time': 'segment_index'})

        asymmetric_signal_segmented = asymmetric_signal_segmented[(segment_length - 1) :][
            :: (segment_length - overlap)
        ].transpose("segment_index", "day_in_segment", "lat", "lon")
        print(rf"{'✔':>1}")

        print(f"{f'→ Detrending data...':<{str_width-1}}", end="")
        # Detrend the data along the segmented axis
        symmetric_signal_detrended = xr.zeros_like((symmetric_signal_segmented))
        asymmetric_signal_detrended = xr.zeros_like((asymmetric_signal_segmented))
        symmetric_signal_detrended[:] = signal.detrend(symmetric_signal_segmented, axis=1)
        asymmetric_signal_detrended[:] = signal.detrend(asymmetric_signal_segmented, axis=1)
        print(rf"{'✔':>1}")

        print(f"{f'→ Windowing data...':<{str_width-1}}", end="")
        # Apply the Hann window to each segment
        symmetric_signal_windowed = xr.zeros_like((symmetric_signal_detrended))
        asymmetric_signal_windowed = xr.zeros_like((asymmetric_signal_detrended))
        symmetric_signal_windowed[:] = np.einsum(
            "j,ijkl->ijkl", hann_window, symmetric_signal_detrended
        )
        asymmetric_signal_windowed[:] = np.einsum(
            "j,ijkl->ijkl", hann_window, asymmetric_signal_detrended
        )
        print(rf"{'✔':>1}")

        print(f"{f'→ Fourier transforming data...':<{str_width-1}}", end="")
        # Fourier transform the data and calculate raw power
        symmetric_signal_fft[variable_id] = xr.DataArray(
            data=np.zeros_like((symmetric_signal_windowed)),
            dims=['segment_index', 'frequency', 'lat', 'wavenumber'],
            coords={
                'segment_index': symmetric_signal_windowed.segment_index,
                'frequency': np.fft.fftshift(np.fft.fftfreq(len(symmetric_signal_windowed.day_in_segment), 1)),
                'lat': symmetric_signal_windowed.lat,
                'wavenumber': np.fft.fftshift(360*np.fft.fftfreq(len(symmetric_signal_windowed.lon), 2.5))[::-1]
            }
        )

        asymmetric_signal_fft[variable_id] = xr.DataArray(
            data=np.zeros_like((symmetric_signal_windowed)),
            dims=['segment_index', 'frequency', 'lat', 'wavenumber'],
            coords={
                'segment_index': symmetric_signal_windowed.segment_index,
                'frequency': np.fft.fftshift(np.fft.fftfreq(len(symmetric_signal_windowed.day_in_segment), 1)),
                'lat': symmetric_signal_windowed.lat,
                'wavenumber': np.fft.fftshift(360*np.fft.fftfreq(len(symmetric_signal_windowed.lon), 2.5))[::-1]
            }
        )

        symmetric_signal_fft[variable_id][:] = (
            np.fft.fftshift(np.fft.fft2(symmetric_signal_windowed, axes=(1, 3)), axes=(1,3))
            / (len(variable.lon) * segment_length)
            * 4
        )

        raw_symmetric_power[variable_id] = (
            (symmetric_signal_fft[variable_id] * symmetric_signal_fft[variable_id].conj()).real.mean(dim=['segment_index', 'lat'])
        )

        asymmetric_signal_fft[variable_id][:] = (
            np.fft.fftshift(np.fft.fft2(asymmetric_signal_windowed, axes=(1, 3)), axes=(1,3))
            / (len(variable.lon) * segment_length)
            * 4
        )

        raw_asymmetric_power[variable_id] = (
            (asymmetric_signal_fft[variable_id] * asymmetric_signal_fft[variable_id].conj()).real.mean(dim=['segment_index', 'lat'])
        )
        print(rf"{'✔':>1}")

        #### 1-2-1 Filtering
        print(f"{f'→ 1-2-1 filtering data...':<{str_width-1}}", end="")
        # Smooths the background spectrum 'n_smooths' times
        background_spectrum[variable_id] = (raw_symmetric_power[variable_id] + raw_asymmetric_power[variable_id]) / 2
        background_spectrum[variable_id][:] = one_two_one_filter(background_spectrum[variable_id].values, n_smooths, "time")
        background_spectrum[variable_id][:] = one_two_one_filter(background_spectrum[variable_id].values, n_smooths, "space")

        # Calculate signal strength as raw/smoothed background
        symmetric_power_spectrum[variable_id].append(
            (raw_symmetric_power[variable_id] / background_spectrum[variable_id]).isel(wavenumber=slice(None, None, -1))
        )
        asymmetric_power_spectrum[variable_id].append(
            (raw_asymmetric_power[variable_id] / background_spectrum[variable_id]).isel(wavenumber=slice(None, None, -1))
        )
        print(rf"{'✔':>1}")
        # print(f"{'-'*str_width}")

symmetric_power_spectrum[variable_id] = xr.concat(symmetric_power_spectrum[variable_id], dim=coords.experiments)
symmetric_power_spectrum[variable_id].attrs['longname'] = variable.name
asymmetric_power_spectrum[variable_id] = xr.concat(asymmetric_power_spectrum[variable_id], dim=coords.experiments)
asymmetric_power_spectrum[variable_id].attrs['longname'] = variable.name

print(f"{'':{'='}^{str_width}}")
print("Finished")

        Space-Time Power Spectra        
             Variable: PRCP             
----------------------------------------
Experiment: -4K                         
→ Defining symmetric signals...        ✔
→ Segmenting data...                   ✔
→ Detrending data...                   ✔
→ Windowing data...                    ✔
→ Fourier transforming data...         

/glade/u/home/sressel/.conda/envs/modified-npl/lib/python3.12/site-packages/xarray/core/indexing.py:1522: ComplexWarning: Casting complex values to real discards the imaginary part
  array[key] = value
/glade/u/home/sressel/.conda/envs/modified-npl/lib/python3.12/site-packages/xarray/core/indexing.py:1522: ComplexWarning: Casting complex values to real discards the imaginary part
  array[key] = value


✔
→ 1-2-1 filtering data...              ✔
Experiment: 0K                          
→ Defining symmetric signals...        ✔
→ Segmenting data...                   ✔
→ Detrending data...                   ✔
→ Windowing data...                    ✔
→ Fourier transforming data...         

/glade/u/home/sressel/.conda/envs/modified-npl/lib/python3.12/site-packages/xarray/core/indexing.py:1522: ComplexWarning: Casting complex values to real discards the imaginary part
  array[key] = value
/glade/u/home/sressel/.conda/envs/modified-npl/lib/python3.12/site-packages/xarray/core/indexing.py:1522: ComplexWarning: Casting complex values to real discards the imaginary part
  array[key] = value


✔
→ 1-2-1 filtering data...              ✔
Experiment: 4K                          
→ Defining symmetric signals...        ✔
→ Segmenting data...                   ✔
→ Detrending data...                   ✔
→ Windowing data...                    ✔
→ Fourier transforming data...         

/glade/u/home/sressel/.conda/envs/modified-npl/lib/python3.12/site-packages/xarray/core/indexing.py:1522: ComplexWarning: Casting complex values to real discards the imaginary part
  array[key] = value


✔
→ 1-2-1 filtering data...              ✔
Finished


/glade/u/home/sressel/.conda/envs/modified-npl/lib/python3.12/site-packages/xarray/core/indexing.py:1522: ComplexWarning: Casting complex values to real discards the imaginary part
  array[key] = value


In [ ]:
east_power = symmetric_power_spectrum['PRCP'].sel(frequency=slice(1/100, 1/20), wavenumber=slice(1,6)).sum(dim=['frequency', 'wavenumber'])
west_power = symmetric_power_spectrum['PRCP'].sel(frequency=slice(1/100, 1/20), wavenumber=slice(-6,-1)).sum(dim=['frequency', 'wavenumber'])
plt.plot(east_power/west_power)
print(east_power.values/west_power.values)

In [ ]:
phase_speed = xr.zeros_like(symmetric_power_spectrum['PRCP'].sel(experiment='0K', drop=True))
phase_speed[:] = np.einsum(
    'i,j->ij',
    frequency/SECONDS_PER_DAY,
    1/(zonal_wavenumber/(2*np.pi*EARTH_RADIUS))
)
weighted_phase_speed = phase_speed * symmetric_power_spectrum['PRCP']
experiment_mean_phase_speed = weighted_phase_speed.sel(
    wavenumber=slice(1,6), frequency=slice(1/100, 1/20)
).mean(dim=['wavenumber', 'frequency'])
print(experiment_mean_phase_speed.values)

weighted_zonal_wavenumber = zonal_wavenumber * symmetric_power_spectrum['PRCP']
experiment_mean_zonal_wavenumber = weighted_zonal_wavenumber.sel(
    wavenumber=slice(1,6), frequency=slice(1/100, 1/20)
).mean(dim=['wavenumber', 'frequency'])
print(experiment_mean_zonal_wavenumber.values)

### Plot space-time power spectra

In [ ]:
variable_to_plot = 'PRCP'

SAVE_FIG = True
PLOT_MODE = "publication"
FIG_LAYOUT = 'twocol'
plt.style.use('bmh')
set_plot_mode(PLOT_MODE)
plt.rcParams['mathtext.fontset'] = 'dejavusans'

fig = plt.figure(figsize=get_figsize(PLOT_MODE, FIG_LAYOUT))
gs = GridSpec(2, 3, height_ratios=[100, 3], figure=fig)
gs.update(bottom=0.25, wspace=0.2, hspace=0.05)

axes = []
axes.append(fig.add_subplot(gs[0, 0]))
axes.append(fig.add_subplot(gs[0, 1]))
axes.append(fig.add_subplot(gs[0, 2]))
cbar_ax = fig.add_subplot(gs[-1, :])

for index, (ax, experiment_to_plot) in enumerate(zip(axes, coords.experiments.values)):

    ax.set_title(
        f"{string.ascii_letters[index]}) {config.EXPERIMENT_DISPLAY_NAMES[experiment_to_plot]}",
        pad=3.5,
        loc='left'
    )
    im = ax.contourf(
        zonal_wavenumber,
        frequency,
        np.log10(symmetric_power_spectrum[variable_to_plot].sel(experiment=experiment_to_plot)),
        norm=mcolors.CenteredNorm(vcenter=0.0),
        cmap=modified_colormap("coolwarm", "white", 0.05, 0.05),
        levels=np.arange(-1, 1+0.1, 0.1),
    )

    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
    cbar.set_ticks(im.levels[::2])
    cbar.set_label(r"log$_{10}$[Symmetric Power (Precipitation)]")

    # Mark zonal wavenumber == 0:
    ax.axvline(x=0, color="#bcbcbc", lw=1, ls="--")
    ax.set_xlim(-10, 10)
    ax.set_xticks([-10, -5, 0, 5, 10])
    ax.set_xlabel("Zonal wavenumber")
    ax.set_ylim(1 / 180, 1 / 5)

    period_ticks = [5, 6.666666666666667, 10, 20, 100]
    frequency_ticks = [1/tick for tick in period_ticks]
    if index == 0:
        ax.set_yticks(ticks=frequency_ticks, labels=np.around(period_ticks, 1))
    else:
        ax.set_yticks(ticks=frequency_ticks, labels=[])
    [ax.axhline(y=freq, color="#bcbcbc", lw=0.75, ls=":") for freq in frequency_ticks]
    ax.set_aspect(20 / (1 / 5 - 1 / 180))
    ax.grid(False)

axes[0].set_ylabel("Period (days)", labelpad=10)

output_filename = 'Space-Time Power Spectra'
logger.info(f"Output directory: {config.AQUAPLANET_OUTPUT_DIRECTORY}/space-time-power-spectra/")
logger.info(f"Output filename: {output_filename}")
if SAVE_FIG:
    print(f"{f'Saving...':<{config.SEP_WIDTH-1}}", end="")
    plt.savefig(
        f"{config.AQUAPLANET_OUTPUT_DIRECTORY}/{output_filename}.pdf",
        dpi=500,
        bbox_inches="tight"
    )
    print(rf"{'✔':>1}")
else:
    print("Not Saving")

plt.show()

In [11]:
symmetric_power_spectrum['PRCP'].name = 'Symmetric Power Spectrum [PRCP]'
symmetric_power_spectrum['PRCP'].to_netcdf("/glade/u/home/sressel/spencer-scratch/aquaplanet_data_for_zenodo/symmetric_power_spectrum_prcp.nc")

## Coherence

### Calculate coherence

In [ ]:
coherence = {}
x_phase = {}
y_phase = {}
statistical_significance_level = {}
alpha = 0.05


for coherence_var1, coherence_var2 in zip(['MSE', 'MSE', 'MSE', 'PRCP', 'PRCP', 'OLR', 'PRCP'], ['PRCP', 'OLR', 'CWV', 'OLR', 'CWV', 'CWV', 'CIT']):
# for coherence_var1, coherence_var2 in zip(['MSE'], ['PRCP']):
    coh_vars = f"{coherence_var1}-{coherence_var2}"
    print(coh_vars)

    cross_spectrum = np.fft.fftshift(
            np.mean(symmetric_signal_fft[coherence_var1] * np.conj(symmetric_signal_fft[coherence_var2]), axis=(0, 2))
        )

    power_coh1 = np.fft.fftshift(
            np.mean(symmetric_signal_fft[coherence_var1] * np.conj(symmetric_signal_fft[coherence_var1]), axis=(0, 2))
        )

    power_coh2 = np.fft.fftshift(
            np.mean(symmetric_signal_fft[coherence_var2] * np.conj(symmetric_signal_fft[coherence_var2]), axis=(0, 2))
        )

    n = np.shape(symmetric_signal_fft[coherence_var1])[0]
    statistical_significance_level[coh_vars] = 1 - alpha ** (1. / (n - 1))

    # Calculate coherence between variables
    coherence[coh_vars] = np.real(np.abs(cross_spectrum)**2/(power_coh1*power_coh2))

    # Calculate phase between variables
    x_phase[coh_vars] = np.imag(cross_spectrum)/np.abs(cross_spectrum)
    y_phase[coh_vars] = np.real(cross_spectrum)/np.abs(cross_spectrum)

    # Remove non-statistically significant points
    non_significant_points = np.where(coherence[coh_vars] <= statistical_significance_level[coh_vars])
    coherence[coh_vars][non_significant_points] = np.nan
    x_phase[coh_vars][non_significant_points] = np.nan
    y_phase[coh_vars][non_significant_points] = np.nan

print("Finished")

In [ ]:
savefig = False
output_directory = f"/glade/u/home/sressel/aquaplanet_analysis/output/"

plt.style.use("bmh")
plt.rcParams.update({"font.size": 14})

fig = plt.figure(figsize=(16, 14))
gs = GridSpec(4, 3, width_ratios = [1, 1, 1], height_ratios=[1, 1, 1, 0.05], figure=fig)
gs.update(top=0.95, bottom=0.05, left=0.05, right=0.95, wspace=0.15, hspace=0.4)

axes = []
axes.append(fig.add_subplot(gs[0,0]))
axes.append(fig.add_subplot(gs[0,1]))
axes.append(fig.add_subplot(gs[0,2]))
axes.append(fig.add_subplot(gs[1,0]))
axes.append(fig.add_subplot(gs[1,1]))
axes.append(fig.add_subplot(gs[1,2]))
axes.append(fig.add_subplot(gs[2,1]))
cbar_ax = fig.add_subplot(gs[3, :])

fig.suptitle(f"Coherence between variables in {experiment_shortname} experiment", x=0.5, y=1.02, fontsize=24)

for index, (ax, coh_vars) in enumerate(zip(axes, coherence.keys())):
    ax.set_facecolor('white')
    ax.set_title(
        rf"{coh_vars.split('-')[0]} & {coh_vars.split('-')[1]}",
        pad=10,
        fontsize=20
    )

    im = ax.contourf(
        zonal_wavenumber,
        -frequency,
        coherence[coh_vars],
        # levels=np.arange(statistical_significance_level[coh_vars], 1.05, 0.05),
        levels=np.arange(0.1, 1.05, 0.05),
        cmap='YlOrRd',
        # extend='max'
    )

    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
    cbar.set_ticks(np.arange(0.1, 1.1, 0.1))
    # cbar.set_title(r"coh$^{2}$")

    precip_im = ax.contour(
        zonal_wavenumber,
        -frequency,
        symmetric_power_spectrum['PRCP'],
        colors='k',
        linewidths=1
    )

    ax.quiver(
        zonal_wavenumber[::1],
        -frequency[::2],
        x_phase[coh_vars][::2, ::1],
        y_phase[coh_vars][::2, ::1],
        scale=15
    )


    # Mark 3, 6, 20 day period:
    # plot_days = [3, 6, 20, 100]
    plot_days = [10, 15, 20, 60, 100]
    for day in plot_days:
        ax.axhline(y=1 / day, color="k", lw=1, ls=":")
        # ax.text(-14.8, 1 / day + 0.005, str(day) + "d", fontsize=15)
        # if index == 2 or index == 5:
        if index > -1:
            ax.text(8.2, 1/day-0.001, str(day) + "d", fontsize=15)

    # # Mark zonal wavenumber == 0:
    # ax.axvline(x=0, color="k", lw=1, ls=":")

    # ax.set_xlim(-15, 15)
    # ax.set_xticks(np.arange(-15, 15, 5))
    # if index > 2:
    #     ax.set_xlabel("Zonal wavenumber")
    # ax.set_ylim(1 / 180, 1 / 2)

    # ax.set_yticks(np.arange(0.1, 0.6, 0.1))
    # if index == 0 or index == 3:
    #     ax.set_ylabel("Frequency")
    # else:
    #     ax.set_yticklabels('')

    # ax.set_aspect(30 / (1 / 2 - 1 / 180))

    # Mark zonal wavenumber == 0:
    ax.axvline(x=0, color="k", lw=1, ls=":")

    ax.set_xlim(-2, 8)
    # ax.set_xticks(np.arange(-15, 15, 5))
    if index > 2:
        ax.set_xlabel("Zonal wavenumber")
    ax.set_ylim(1 / 180, 1 / 10)

    # ax.set_yticks(np.arange(0.1, 0.6, 0.1))
    if index == 0 or index == 3:
        ax.set_ylabel("Frequency")
    else:
        ax.set_yticklabels('')

    ax.set_aspect('auto')
    # ax.set_aspect(30 / (1 / 2 - 1 / 180))

if not savefig:
    plt.show()
else:
    save_string = (
        f"symmetric_{experiment}"
      + f"_multi-variable_coherence_with-precip.png"
        )
    print(f"Saving plot as {save_string}")
    plt.savefig(
        f"{output_directory}/coherence/{save_string}",
        dpi=500,
        bbox_inches="tight",
    )

print(f"{'='*40}")
print("Finished")